<a href="https://colab.research.google.com/github/shaguftaraihaan/first-one/blob/main/app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q \
    diffusers==0.20.0 \
    transformers==4.30.2 \
    huggingface_hub==0.15.1 \
    accelerate==0.20.3 \
    xformers \
    opencv-python-headless \
    Pillow \
    gradio==3.50.2 \
    torch torchvision --quiet
    pip install scikit-image

print('✅ All packages installed!')

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected : {gpu_name}')
    print(f'   VRAM         : {vram:.1f} GB')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    DPMSolverMultistepScheduler,
    StableDiffusionPipeline,
)

# ── Device ──────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32
print(f'[INFO] Device : {device}  |  dtype : {dtype}')

# ── ControlNet ───────────────────────────────────────────────
print('[INFO] Loading ControlNet (Canny) …')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/sd-controlnet-canny',
    torch_dtype=dtype,
)

# ── ControlNet + SD Pipeline ─────────────────────────────────
print('[INFO] Loading Stable Diffusion v1.5 + ControlNet pipeline …')
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
)

# GPU memory optimisations
if device == 'cuda':
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print('[INFO] xformers attention enabled ✅')
    except Exception:
        pipe.enable_attention_slicing()
        print('[INFO] Attention slicing enabled (xformers unavailable)')
else:
    pipe.enable_attention_slicing()

# ── Text-to-Image fallback (shares weights via VAE, saves VRAM) ──
print('[INFO] Loading text-to-image fallback …')
txt2img_pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None,
).to(device)
txt2img_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    txt2img_pipe.scheduler.config,
    use_karras_sigmas=True,
)
if device == 'cuda':
    try:
        txt2img_pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        txt2img_pipe.enable_attention_slicing()
else:
    txt2img_pipe.enable_attention_slicing()

print('[INFO] ✅ All models loaded and ready!')

# ════════════════════════════════════════════════════════════
#  STYLE & TONE CONFIGS
# ════════════════════════════════════════════════════════════

STYLES = {
    'Cinematic 🎬': {
        'positive'        : 'cinematic lighting, dramatic shadows, golden hour, film still, anamorphic lens, moody atmosphere, bokeh, color graded',
        'negative'        : 'flat lighting, overexposed, washed out, cartoon, illustration, low quality, blurry, watermark',
        'guidance'        : 12.0,
        'controlnet_scale': 0.45,
    },
    'Artistic 🎨': {
        'positive'        : 'oil painting, impasto technique, expressive brush strokes, highly artistic, painterly texture, fine art, museum quality',
        'negative'        : 'photograph, realistic, 3d render, ugly, deformed, bad anatomy, watermark',
        'guidance'        : 10.0,
        'controlnet_scale': 0.5,
    },
    'Realistic 📸': {
        'positive'        : 'ultra realistic photograph, DSLR, natural lighting, sharp focus, 8k resolution, photorealistic, subsurface scattering',
        'negative'        : 'painting, illustration, cartoon, sketch, anime, blurry, low quality, watermark, bad anatomy',
        'guidance'        : 13.0,
        'controlnet_scale': 0.6,
    },
    'Fantasy ✨': {
        'positive'        : 'fantasy art, magical glow, ethereal lighting, mystical atmosphere, epic scene, unreal engine, concept art, artstation trending',
        'negative'        : 'mundane, modern, photograph, realistic, ugly, bad anatomy, watermark, low quality',
        'guidance'        : 11.0,
        'controlnet_scale': 0.4,
    },
    'Anime 🌸': {
        'positive'        : 'anime style, studio ghibli, cel shading, vibrant anime colors, detailed linework, manga art, beautiful anime illustration',
        'negative'        : 'realistic, photograph, 3d render, ugly, bad anatomy, watermark, low quality',
        'guidance'        : 10.5,
        'controlnet_scale': 0.55,
    },
    'Cyberpunk ⚡': {
        'positive'        : 'cyberpunk, neon lights, futuristic city, blade runner aesthetic, rain reflections, high contrast neon, sci-fi, digital art',
        'negative'        : 'natural, rural, daylight, low quality, blurry, watermark, bad anatomy',
        'guidance'        : 11.5,
        'controlnet_scale': 0.45,
    },
}

COLOR_TONES = {
    'None'       : '',
    'Warm 🔥'    : 'warm tones, golden light, amber, sepia warmth',
    'Cool 🧊'    : 'cool tones, blue tint, winter light, desaturated blues',
    'Vibrant 🌈' : 'vibrant saturated colors, neon accents, vivid palette',
    'Moody 🌑'   : 'dark moody tones, deep shadows, low key lighting, noir',
    'Pastel 🌸'  : 'soft pastel colors, gentle tones, light airy palette',
}

# ════════════════════════════════════════════════════════════
#  CORE FUNCTIONS
# ════════════════════════════════════════════════════════════

def extract_edges(pil_image, low_thresh=80, high_thresh=200):
    img_np  = np.array(pil_image.resize((512, 512)).convert('RGB'))
    blurred = cv2.GaussianBlur(img_np, (5, 5), 0)
    gray    = cv2.cvtColor(blurred, cv2.COLOR_RGB2GRAY)
    edges   = cv2.Canny(gray, low_thresh, high_thresh)
    kernel  = np.ones((2, 2), np.uint8)
    edges   = cv2.dilate(edges, kernel, iterations=1)
    return Image.fromarray(np.stack([edges] * 3, axis=-1))

def post_process(image, style):
    if   style == 'Cinematic 🎬':  image = ImageEnhance.Contrast(image).enhance(1.15); image = ImageEnhance.Color(image).enhance(0.9)
    elif style == 'Artistic 🎨':   image = image.filter(ImageFilter.SMOOTH); image = ImageEnhance.Color(image).enhance(1.3)
    elif style == 'Realistic 📸':  image = ImageEnhance.Sharpness(image).enhance(1.4)
    elif style == 'Fantasy ✨':     image = ImageEnhance.Color(image).enhance(1.5); image = ImageEnhance.Brightness(image).enhance(1.05)
    elif style == 'Anime 🌸':      image = ImageEnhance.Color(image).enhance(1.4); image = ImageEnhance.Sharpness(image).enhance(1.2)
    elif style == 'Cyberpunk ⚡':  image = ImageEnhance.Color(image).enhance(1.6); image = ImageEnhance.Contrast(image).enhance(1.2)
    return image

def build_prompt(user_prompt, style, color_tone):
    preset = STYLES[style]
    tone_mod = COLOR_TONES.get(color_tone, '')
    parts    = [p for p in [user_prompt, preset['positive'], tone_mod] if p]
    positive = ', '.join(parts) + ', masterpiece, best quality, ultra detailed, 8k, sharp focus'
    negative = preset['negative'] + ', ugly, tiling, poorly drawn hands, poorly drawn feet, poorly drawn face, out of frame, extra limbs, disfigured, deformed, blurry, bad anatomy, watermark, grainy, signature, cut off'
    return positive, negative

def generate_from_image(input_image, prompt, style, color_tone, steps, seed):
    cfg     = STYLES[style]
    pos, neg = build_prompt(prompt, style, color_tone)
    edge    = extract_edges(input_image)
    gen     = None if seed < 0 else torch.Generator(device).manual_seed(int(seed))
    result  = pipe(
        prompt=pos, negative_prompt=neg, image=edge,
        num_inference_steps=int(steps),
        guidance_scale=cfg['guidance'],
        controlnet_conditioning_scale=cfg['controlnet_scale'],
        generator=gen,
    ).images[0]
    return input_image, edge, post_process(result, style), pos

def generate_from_text(prompt, style, color_tone, steps, seed):
    pos, neg = build_prompt(prompt, style, color_tone)
    gen      = None if seed < 0 else torch.Generator(device).manual_seed(int(seed))
    result   = txt2img_pipe(
        prompt=pos, negative_prompt=neg,
        num_inference_steps=int(steps),
        guidance_scale=STYLES[style]['guidance'],
        generator=gen, height=512, width=512,
    ).images[0]
    blank = Image.new('RGB', (512, 512), (20, 20, 30))
    return blank, blank, post_process(result, style), pos

def process_input(sketch, uploaded_image, prompt, style, color_tone, steps, seed):
    try:
        if uploaded_image is not None:
            img = uploaded_image.convert('RGB')
            o, e, r, p = generate_from_image(img, prompt, style, color_tone, steps, seed)
        elif sketch is not None:
            composite = sketch.get('composite') if isinstance(sketch, dict) else None
            if composite is None:
                return None, None, None, '⚠️ Please finish drawing your sketch!'
            img = Image.fromarray(composite.astype('uint8')).convert('RGB')
            o, e, r, p = generate_from_image(img, prompt, style, color_tone, steps, seed)
        elif prompt and prompt.strip():
            o, e, r, p = generate_from_text(prompt.strip(), style, color_tone, steps, seed)
        else:
            return None, None, None, '❌ Please draw a sketch, upload an image, or enter a text prompt.'
        return o, e, r, f'✅ Done!\n\n📝 Prompt used:\n{p}'
    except Exception as exc:
        import traceback
        return None, None, None, f'❌ Error: {exc}\n\n{traceback.format_exc()}'

print('✅ All functions defined — ready to launch UI!')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[INFO] Device : cuda  |  dtype : torch.float16
[INFO] Loading ControlNet (Canny) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

[INFO] Loading Stable Diffusion v1.5 + ControlNet pipeline …


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 4 — Launch Gradio UI  (get your public link)   ║
# ╚══════════════════════════════════════════════════════╝

import gradio as gr

# ── Custom CSS ───────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Sans:wght@300;400;500&display=swap');

:root {
    --bg-base:   #070b14;
    --bg-card:   rgba(255,255,255,0.04);
    --bg-card2:  rgba(255,255,255,0.07);
    --border:    rgba(255,255,255,0.08);
    --accent:    #8b5cf6;
    --accent2:   #ec4899;
    --accent3:   #06b6d4;
    --text-hi:   #f1f5f9;
    --text-mid:  #94a3b8;
    --radius:    16px;
    --glow:      0 0 40px rgba(139,92,246,0.25);
}

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container {
    background: var(--bg-base) !important;
    font-family: 'DM Sans', sans-serif !important;
    color: var(--text-hi) !important;
}

.gradio-container::before {
    content: '';
    position: fixed; inset: 0;
    background:
        radial-gradient(ellipse 80% 60% at 20% 0%,  rgba(139,92,246,0.18) 0%, transparent 60%),
        radial-gradient(ellipse 60% 50% at 80% 100%,rgba(6,182,212,0.14)  0%, transparent 60%),
        radial-gradient(ellipse 50% 40% at 50% 50%, rgba(236,72,153,0.08) 0%, transparent 60%);
    pointer-events: none; z-index: 0;
}

.app-header { text-align:center; padding:2.5rem 1rem 1rem; }
.app-header h1 {
    font-family:'Syne',sans-serif !important;
    font-size:clamp(2rem,5vw,3.2rem) !important;
    font-weight:800 !important;
    background:linear-gradient(135deg,#a78bfa 0%,#ec4899 50%,#06b6d4 100%);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    background-clip:text; letter-spacing:-0.03em; margin:0 !important;
}
.app-header p { color:var(--text-mid) !important; font-size:0.95rem !important; margin-top:0.6rem !important; }
.badge {
    display:inline-block; padding:0.3rem 1rem; border-radius:999px;
    border:1px solid rgba(139,92,246,0.5);
    background:rgba(139,92,246,0.12); color:#c4b5fd;
    font-size:0.72rem; letter-spacing:0.15em; text-transform:uppercase; margin-bottom:0.8rem;
}
.gpu-badge {
    display:inline-block; padding:0.25rem 0.85rem; border-radius:999px;
    border:1px solid rgba(6,182,212,0.5);
    background:rgba(6,182,212,0.1); color:#67e8f9;
    font-size:0.72rem; letter-spacing:0.12em; text-transform:uppercase; margin-left:0.5rem;
}

.glass-card {
    background:var(--bg-card) !important;
    backdrop-filter:blur(20px) !important;
    border:1px solid var(--border) !important;
    border-radius:24px !important;
    padding:1.5rem !important;
    transition:border-color 0.3s, box-shadow 0.3s;
}
.glass-card:hover { border-color:rgba(139,92,246,0.3) !important; box-shadow:var(--glow) !important; }

label, .label-wrap span {
    font-family:'Syne',sans-serif !important;
    font-size:0.72rem !important; font-weight:700 !important;
    letter-spacing:0.12em !important; text-transform:uppercase !important;
    color:var(--text-mid) !important;
}

textarea, input[type=text], input[type=number] {
    background:rgba(255,255,255,0.04) !important;
    border:1px solid var(--border) !important;
    border-radius:12px !important; color:var(--text-hi) !important;
    font-family:'DM Sans',sans-serif !important;
    transition:border-color 0.2s, box-shadow 0.2s !important;
}
textarea:focus, input:focus {
    border-color:var(--accent) !important;
    box-shadow:0 0 0 3px rgba(139,92,246,0.2) !important; outline:none !important;
}

#generate-btn {
    background:linear-gradient(135deg,#7c3aed 0%,#db2777 100%) !important;
    border:none !important; border-radius:14px !important; color:white !important;
    font-family:'Syne',sans-serif !important; font-size:1rem !important;
    font-weight:700 !important; letter-spacing:0.05em !important;
    padding:0.875rem 2rem !important; width:100% !important;
    transition:opacity 0.2s, transform 0.15s, box-shadow 0.2s !important;
    box-shadow:0 4px 24px rgba(139,92,246,0.4) !important; cursor:pointer;
}
#generate-btn:hover  { opacity:0.9; transform:translateY(-2px); box-shadow:0 8px 32px rgba(139,92,246,0.55) !important; }
#generate-btn:active { transform:translateY(0); }

.output-image img {
    border-radius:14px !important;
    border:1px solid var(--border) !important;
    transition:transform 0.3s !important;
}
.output-image img:hover { transform:scale(1.01); }

#status-box textarea {
    background:rgba(6,182,212,0.06) !important;
    border:1px solid rgba(6,182,212,0.25) !important;
    border-radius:12px !important; color:#67e8f9 !important;
    font-family:'DM Sans',monospace !important; font-size:0.82rem !important;
}

.divider { border:none; border-top:1px solid var(--border); margin:1rem 0; }

input[type=range] { accent-color:var(--accent) !important; }

@media(max-width:768px) {
    .app-header h1 { font-size:1.8rem !important; }
    .glass-card { padding:1rem !important; }
}
"""

# ── Detect GPU for header badge ──────────────────────────────
import torch
gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
header_html = f"""
<div class="app-header">
  <div class="badge">✦ AI-Powered Art Generator</div>
  <span class="gpu-badge">⚡ {gpu_info}</span>
  <h1>Pose to Masterpiece</h1>
  <p>Draw a sketch · Upload an outline · Or just type an idea — AI does the rest</p>
</div>
"""

# ── Build UI ─────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.violet,
        neutral_hue=gr.themes.colors.slate,
        font=[gr.themes.GoogleFont('DM Sans'), 'sans-serif'],
    ),
    css=CSS,
    title='Pose to Masterpiece Generator',
) as demo:

    gr.HTML(header_html)

    with gr.Row(equal_height=False):

        # ── LEFT: Inputs ─────────────────────────────────────
        with gr.Column(scale=1, min_width=320, elem_classes='glass-card'):

            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.75rem">Step 1 — Input</p>')

            with gr.Tabs():
                with gr.TabItem('✏️  Draw Sketch'):
                    sketch_input = gr.Sketchpad(
                        label='Draw your pose or outline',
                        height=300,
                    )
                with gr.TabItem('📤  Upload Image'):
                    upload_input = gr.Image(
                        label='Upload outline / sketch / reference',
                        type='pil', height=300,
                    )
                with gr.TabItem('💬  Text Only'):
                    gr.HTML('<p style="color:#94a3b8;font-size:0.82rem;margin:0.75rem 0 0.25rem">No sketch? Just type a prompt below and generate purely from text.</p>')

            gr.HTML('<hr class="divider"/>')
            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.4rem">Step 2 — Describe your scene</p>')

            prompt_input = gr.Textbox(
                label='',
                placeholder='e.g. "Indian cricketer in blue jersey hitting a six, golden hour"',
                lines=3, show_label=False,
            )

            gr.HTML('<hr class="divider"/>')
            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.4rem">Step 3 — Style</p>')

            style_input = gr.Radio(
                choices=list(STYLES.keys()),
                value='Cinematic 🎬', label='',
            )

            gr.HTML('<hr class="divider"/>')
            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.4rem">Step 4 — Color Tone</p>')

            color_tone_input = gr.Radio(
                choices=list(COLOR_TONES.keys()),
                value='None', label='',
            )

            gr.HTML('<hr class="divider"/>')
            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.4rem">Advanced</p>')

            with gr.Row():
                steps_input = gr.Slider(
                    minimum=15, maximum=60, value=30, step=1,
                    label='Inference Steps  (more = better quality, slower)',
                )
            with gr.Row():
                seed_input = gr.Number(
                    value=-1, label='Seed  (-1 = random)', precision=0,
                )

            generate_btn = gr.Button(
                '⚡  Generate Masterpiece',
                elem_id='generate-btn', variant='primary',
            )

        # ── RIGHT: Outputs ────────────────────────────────────
        with gr.Column(scale=1, min_width=320, elem_classes='glass-card'):

            gr.HTML('<p style="font-family:Syne,sans-serif;font-size:0.7rem;letter-spacing:0.12em;text-transform:uppercase;color:#94a3b8;margin:0 0 0.75rem">Output</p>')

            with gr.Row():
                original_output = gr.Image(
                    label='🖼 Input Preview',
                    elem_classes='output-image', height=210,
                )
                edge_output = gr.Image(
                    label='📐 Edge Map',
                    elem_classes='output-image', height=210,
                )

            final_output = gr.Image(
                label='✨ AI Masterpiece',
                elem_classes='output-image', height=370,
                show_download_button=True,
            )

            status_output = gr.Textbox(
                label='Status & Prompt Used',
                interactive=False, lines=4,
                elem_id='status-box',
            )

    gr.HTML('<div style="text-align:center;padding:1.5rem 1rem 0.75rem;color:#475569;font-size:0.78rem;">Built with ❤️ using Stable Diffusion + ControlNet &nbsp;·&nbsp; <span style="color:#7c3aed">Pose to Masterpiece Generator</span></div>')

    generate_btn.click(
        fn=process_input,
        inputs=[sketch_input, upload_input, prompt_input,
                style_input, color_tone_input, steps_input, seed_input],
        outputs=[original_output, edge_output, final_output, status_output],
    )

# ── Launch with public Gradio share link ─────────────────────
demo.launch(share=True, debug=False)
# share=True gives you a public https://xxxxx.gradio.live link
# that works in any browser — even outside Colab!


In [ ]:
pipe = StableDiffusionControlNetPipeline.from_pretrained(...)
pipe.save_pretrained("my_model")